In [1]:
import re, json
from pathlib import Path
import requests
import unicodedata, pymupdf
from collections import defaultdict, Counter


DATA = Path("Data")
LAW  = "قانون حماية البيانات الشخصية 151 لسنة 2020"
LAW2 = "قانون حماية المستهلك 181 لسنة 2018"
LAW3 = "قانون العمل 14 لسنة 2025"
CODES = {LAW: "pdp", LAW2: "cpl", LAW3: "labor"}

for f in sorted(DATA.rglob("*")):
    if f.is_file():
        print(f.relative_to(DATA), f.stat().st_size // 1024, "KB")

consumer_protection_181_2018.txt 70 KB
labor_law_raw_wikitext.txt 181 KB
personal_data_151_2020.txt 64 KB
processed\chunks.json 544 KB
processed\corpus_articles.json 536 KB
processed\evaluation_results.csv 11 KB
qnwn_lml_ljdyd_14_lsn_2025_m_q.pdf 934 KB


In [2]:
LAW2 = "قانون حماية المستهلك 181 لسنة 2018"
t2 = Path("Data/consumer_protection_181_2018.txt").read_text(encoding="utf-8")

c_art_re  = re.compile(r"^المادة\s+(\d+)\s*$")
c_head_re = re.compile(r"^(الباب|الفصل)\s+[^\s:]+\s*:")

consumer = []
part, bab, fasl, cur = "issuance", "", "", None

for line in t2.splitlines():
    s = line.strip()
    if not s:
        continue
    if s == "قانون حماية المستهلك":
        part, bab, fasl, cur = "law", "", "", None
        continue
    m = c_art_re.match(s)
    if m:
        chapter = "مواد الإصدار" if part == "issuance" else " - ".join(x for x in (bab, fasl) if x)
        cur = {"law": LAW2, "article_type": part, "article": int(m.group(1)),
               "chapter": chapter, "lines": []}
        consumer.append(cur)
        continue
    h = c_head_re.match(s)
    if h:
        if h.group(1) == "الباب":
            bab, fasl = s, ""
        else:
            fasl = s
        cur = None
        continue
    if cur is not None:
        cur["lines"].append(s)

for r in consumer:
    r["text"] = "\n".join(r.pop("lines"))

print(len(consumer), "records |", sum(r["article_type"] == "law" for r in consumer), "law articles")
r30 = next(r for r in consumer if r["article_type"] == "law" and r["article"] == 30)
print(r30["chapter"])
print(r30["text"][:100])
r76 = next(r for r in consumer if r["article_type"] == "law" and r["article"] == 76)
print("art 76 ends:", r76["text"][-40:])

81 records | 76 law articles
الباب الثاني: التزامات المورد والمعلن - الفصل الثاني: أحكام خاصة ببعض التعاقدات
يلتزم المورد في حالة البيع بالتقسيط بتسليم المستهلك فاتورة أو محررا يشمل البيانات الآتية:
1- السعر ا
art 76 ends: بطلب كتابي من الوزير المختص أو من يفوضه.


In [3]:
raw_path = DATA / "labor_law_raw_wikitext.txt"
if not raw_path.exists():
    r = requests.get(
        "https://ar.wikisource.org/w/index.php",
        params={"title": "قانون العمل 14 لسنة 2025 - مصر", "action": "raw"},
        headers={"User-Agent": "rag-student-project/1.0 (personal study project)"},
        timeout=30,
    )
    r.raise_for_status()
    raw_path.write_text(r.text, encoding="utf-8")
print(raw_path.stat().st_size, "bytes")

185644 bytes


In [4]:
t = (DATA / "personal_data_151_2020.txt").read_text(encoding="utf-8")

art_re  = re.compile(r"(?<![\u0621-\u064A])مادة\s*\(\s*(\d+)\s*\)\s*:")
chap_re = re.compile(r"\(\s*(الفصل [^)]+?)\s*\)")
iss_re  = re.compile(r"\(\s*المادة\s+(\S+)\s*\)")
tail_re = re.compile(r"\s*(?:(?:أولاً|أولا|ثانيا|ثالثا|رابعا|خامسا)\s*:[^\n.]*|الصلح والتصالح)\s*$")

def tidy(s):
    s = re.sub(r"[ \t]+", " ", s).strip()
    return tail_re.sub("", s).strip()

idx = t.index("( الفصل الأول )")
issuing, law = t[:idx], t[idx:]

records = []
ms = list(art_re.finditer(law))
for i, m in enumerate(ms):
    end = ms[i + 1].start() if i + 1 < len(ms) else len(law)
    body = law[m.end():end]
    cm = chap_re.search(body)
    if cm:
        body = body[:cm.start()]
    chapter = [c.group(1) for c in chap_re.finditer(law[:m.start()])][-1]
    records.append({"law": LAW, "article_type": "law", "article": int(m.group(1)),
                    "chapter": chapter, "text": tidy(body)})

im = list(iss_re.finditer(issuing))
for i, m in enumerate(im):
    end = im[i + 1].start() if i + 1 < len(im) else len(issuing)
    records.append({"law": LAW, "article_type": "issuance", "article": m.group(1),
                    "chapter": "مواد الإصدار", "text": tidy(issuing[m.end():end])})

print(len(records), "records")
print(records[4]["chapter"], "|", records[4]["text"][:80])
print("art 48 ends:", records[47]["text"][-50:])

56 records
الفصل الثالث | مع مراعاة أحكام المادة (12) من هذا القانون ، يلتزم معالج البيانات الشخصية بما يأ
art 48 ends: وص عليها في هذا القانون بنصف العقوبة المقررة لها .


In [5]:
w = (DATA / "labor_law_raw_wikitext.txt").read_text(encoding="utf-8")

for m in list(re.finditer("…", w))[::18][:6]:
    print("…", w[max(0, m.start() - 60): m.end() + 40].replace("\n", " "))

w = w[w.index("== نص القانون =="):]
if "== المراجع ==" in w:
    w = w[:w.index("== المراجع ==")]

AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def wiki_clean(s):
    s = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", s)
    s = re.sub(r"\{\{[^{}]*\}\}", "", s)
    s = re.sub(r"'{2,}", "", s)
    s = re.sub(r"<[^>]+>", "", s)
    s = s.translate(AR2EN)
    s = re.sub(r"^-\s*(\d+)\s", r"\1- ", s.strip())
    return re.sub(r"[ \t]+", " ", s).strip()

head_re    = re.compile(r"^(=+)\s*(.+?)\s*=+\s*$")
law_art_re = re.compile(r"^المادة\s+(\d+)$")
iss_art_re = re.compile(r"^المادة\s+([^\d]+)$")
sect_re    = re.compile(r"^(الفصل|القسم|الفرع)\s")

labor, bab, sect, cur = [], "", "", None
for line in w.splitlines():
    s = line.strip()
    if not s:
        continue
    h = head_re.match(s)
    if h:
        title = wiki_clean(h.group(2))
        m = law_art_re.match(title)
        if m:
            cur = {"law": LAW3, "article_type": "law", "article": int(m.group(1)),
                   "chapter": " - ".join(x for x in (bab, sect) if x), "lines": []}
            labor.append(cur); continue
        m = iss_art_re.match(title)
        if m:
            cur = {"law": LAW3, "article_type": "issuance", "article": m.group(1).strip(),
                   "chapter": "مواد الإصدار", "lines": []}
            labor.append(cur); continue
        if title.startswith("الباب"):
            bab, sect = title, ""
        elif sect_re.match(title):
            sect = title
        cur = None
        continue
    if cur is not None:
        cur["lines"].append(wiki_clean(s))

for r in labor:
    r["text"] = "\n".join(l for l in r.pop("lines") if l)

nums = sorted(r["article"] for r in labor if r["article_type"] == "law")
print(len(labor), "records |", len(nums), "law articles | max:", nums[-1])
print("missing:", sorted(set(range(1, nums[-1] + 1)) - set(nums)))
print("empty:", [r["article"] for r in labor if not r["text"]][:20])
print("issuance:", [r["article"] for r in labor if r["article_type"] == "issuance"])
r194 = next(r for r in labor if r["article_type"] == "law" and r["article"] == 194)
print(r194["chapter"], "\n", r194["text"])

… ، شرط التحكيم، مشارطة التحكيم، الإضراب عن العمل، إصابة العمل… إلخ.  === المادة ٢ === تعتبر السنة ٣٦٥ 
… = يجوز للمجلس الأعلى تشكيل مجالس تنفيذية على مستوى المحافظات…  === المادة ٢٠ === يمارس «صندوق تمويل ا
…  المهنة أو الحرفة… رسوم لا تجاوز ٥٠٠ جنيه… شهادة مستوى مهارة… ويُستثنى خريجو التعليم الفني والعالي في
… غيل الخاصة (الملكية، رأس المال، خطاب ضمان مليون جنيه… الرسوم… حالات الوقف).  === المادة ٤٢ === التزام
… تزام منشآت مائة عاملة فأكثر بإنشاء دار حضانة أو التعاقد معها… مع إمكان تحمل التكاليف بدلًا من الإنشاء
… سوم وفئات) تشمل: نسب من الأجور الفعلية في المقاولات والمناجم… اشتراكات قيد لفئات أخرى… نسبة من مبيعات
301 records | 288 law articles | max: 298
missing: [72, 73, 74, 75, 76, 77, 78, 79, 80, 200]
empty: []
issuance: ['الأولى', 'الثانية', 'الثالثة', 'الرابعة', 'الخامسة', 'السادسة', 'السابعة', 'الثامنة', 'التاسعة', 'العاشرة', 'الحادية عشرة', 'الثانية عشرة', 'الثالثة عشرة']
الباب الرابع: المفاوضة الجماعية واتفاقيات العمل الجماعية 
 تُجرى المفاوضة الجماعية بحرية وطواعية في إطار 

In [6]:
import re, unicodedata, pymupdf
from collections import defaultdict, Counter

PDF = DATA / "qnwn_lml_ljdyd_14_lsn_2025_m_q.pdf"
AR_DIG = "٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹"
DIG = str.maketrans(AR_DIG, "01234567890123456789")
YA  = str.maketrans({"ی": "ي", "ک": "ك"})
DIGCH = re.compile(f"[0-9{AR_DIG}]")

def norm(s):
    s = unicodedata.normalize("NFKC", s)
    s = re.sub("[\u0640\u200e\u200f\u202a-\u202e\u2066-\u2069]", "", s)
    return s.translate(DIG).translate(YA)

def digit_chars(page):
    """كل رقم على الصفحة مع مكانه"""
    out = []
    for b in page.get_text("rawdict")["blocks"]:
        for l in b.get("lines", []):
            for s in l["spans"]:
                for ch in s["chars"]:
                    c = unicodedata.normalize("NFKC", ch["c"])
                    if len(c) == 1 and DIGCH.fullmatch(c):
                        x0, y0, x1, y1 = ch["bbox"]
                        out.append(((x0 + x1) / 2, (y0 + y1) / 2, c))
    return out

STATS = Counter()

def fix_digits(w, dch):
    """رتّب أرقام الكلمة من الشمال لليمين حسب مكانها"""
    t = unicodedata.normalize("NFKC", w[4])
    pos = [i for i, c in enumerate(t) if DIGCH.fullmatch(c)]
    if len(pos) < 2:
        return t
    inside = sorted((cx, c) for cx, cy, c in dch
                    if w[0] - 0.7 <= cx <= w[2] + 0.7 and w[1] - 0.7 <= cy <= w[3] + 0.7)
    if len(inside) != len(pos):
        STATS["mismatch"] += 1
        return t
    STATS["fixed"] += 1
    t = list(t)
    for i, (_, c) in zip(pos, inside):
        t[i] = c
    return "".join(t)

GAPS = []

def page_rows(page, join_gap=0.12, collect=False):
    dch = digit_chars(page)
    words = [(w[0], w[1], w[2], w[3], fix_digits(w, dch)) for w in page.get_text("words")]
    words.sort(key=lambda w: (w[1] + w[3]) / 2)
    rows, cur = [], []
    for w in words:
        y, h = (w[1] + w[3]) / 2, (w[3] - w[1]) or 12
        if cur and abs(y - sum((t[1] + t[3]) / 2 for t in cur) / len(cur)) > 0.6 * h:
            rows.append(cur); cur = []
        cur.append(w)
    if cur:
        rows.append(cur)
    lines = []
    for row in rows:
        row = sorted(row, key=lambda t: -t[2])           # من اليمين لليسار
        out = norm(row[0][4])
        for a, b in zip(row, row[1:]):
            gap = a[0] - b[2]
            h = max(a[3] - a[1], b[3] - b[1]) or 12
            if collect:
                GAPS.append(gap / h)
            out += ("" if gap < join_gap * h else " ") + norm(b[4])
        lines.append(out.strip())
    return lines

HDR = re.compile(r"الجر\s*ي?\s*دة|الرسمية|العدد|تابع|مايو|^\s*سنة\s*$|^\s*20\d\d\s*$|^\s*\d{1,3}\s*$")

def strip_header(lines):
    head = [l for l in lines[:8]
            if not (re.search(r"الجر\s*ي?\s*دة\s*الرسمية", l) or (len(l) < 40 and HDR.search(l)))]
    return head + lines[8:]

pdf = pymupdf.open(PDF)
full = "\n".join("\n".join(strip_header(page_rows(p, collect=(i < 6)))) for i, p in enumerate(pdf))
print(len(pdf), "pages |", len(full), "chars")
print("digit words:", dict(STATS))

# عناوين المواد (بتقبل مادة)3:( ومادة)85( ومادة 85 :)
mark = re.compile(r"(?<![\u0621-\u064A])مادة\s*[()]?\s*(\d{1,3})\s*:?\s*[()]")
marks, expected = [], 1
for m in mark.finditer(full):
    v, r = int(m.group(1)), int(m.group(1)[::-1])
    ok = [x for x in (v, r) if expected <= x <= expected + 8]
    if ok:
        n = min(ok, key=lambda x: x - expected)
        marks.append((n, m.start(), m.end()))
        expected = n + 1

nums = [x[0] for x in marks]
print("headings:", len(marks), "| max:", nums[-1])
print("missing:", sorted(set(range(1, nums[-1] + 1)) - set(nums)))

STRUCT = re.compile(r"^[\s()]*(الكتاب|الباب|الفصل|الفرع)\s+\S+[\s()]*$", re.M)

def art_text(n):
    i = nums.index(n)
    end = marks[i + 1][1] if i + 1 < len(marks) else len(full)
    t = full[marks[i][2]:end]
    s = STRUCT.search(t)
    return (t[:s.start()] if s else t).strip(" :\n")

for n in (2, 3, 41, 114):
    print(f"\n=== مادة {n} ===\n{art_text(n)[:350]}")


110 pages | 147413 chars
digit words: {'fixed': 912}
headings: 298 | max: 298
missing: []

=== مادة 2 ===
فى تطبيق أحكام هذا القانون تعتبر السنة ) 365( ايوم ، والشهر ثلاثونيوما ما لم يتم
الاتفاق على خلاف ذلك.

=== مادة 3 ===
ي.عتبر هذا القانون هو القانون العام الذى يحكم علاقات العمل

=== مادة 41 ===
مع عدم الإخلال بالشروط التى يوجبها قانون شركات المساهمة وشركات
التوصية بالأسهم والشركات ذات المسئولية المحدودة وشركات الشخص
الواحد الصادر بالقانون رقم 159 لسنة 1981 يلزم للحصول على الترخيص المشار
إليه فى البند)3( من المادة)40( من هذا القانون ، توافر الشروط المقررة لذلك ،
وعلى الأخص :
1- أن يكون المؤسسون وأعضاء مجلس الإدارة والمديرون المختصون بعمليا

=== مادة 114 ===
مع عدم الإخلال بأحكام قانون تنظيم بعض أوضاع وإجراءات التقاضى فى
مسائل الأحوال الشخصية الصادر بالقانون رقم 1 لسنة 2000 ، لا يجوز فى جميع
الأحوال الاستقطاع أو الحجز ، أو النزول عن الأجر المستحق للعامل لأداء أى دين
إلا فى حدود خمسة وعشرين بالمائة من هذا الأجر ، ويجوز رفع نسبة الخصم إلى
خمسين بالمائة فى حالة دين النفقة.
وعندالتز

In [7]:
from collections import Counter

if "labor_wiki" not in globals():
    labor_wiki = list(labor)

def canon(s):
    s = re.sub(r"[\u064B-\u0652\u0670\u0640]", "", norm(s))
    s = s.translate(str.maketrans("أإآىةؤئ", "ااايهوي"))
    return re.sub(r"[^\u0621-\u064A0-9]", "", s)

def grams(s, k=5):
    return Counter(s[i:i + k] for i in range(len(s) - k + 1))

def sim(a, b):
    ga, gb = grams(a), grams(b)
    return 2 * sum((ga & gb).values()) / max(1, sum(ga.values()) + sum(gb.values()))

def unwrap(t):
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n(?!\d{1,2}\s*[-–.])", " ", t)
    return re.sub(r" +", " ", t).strip()

wiki     = {r["article"]: r for r in labor_wiki if r["article_type"] == "law"}
issuance = [r for r in labor_wiki if r["article_type"] == "issuance"]
pdf_txt  = {n: art_text(n) for n in nums}

sims, final, last_chapter = {}, [], ""
for n in range(1, 299):
    w, p = wiki.get(n), pdf_txt.get(n)
    s = sim(canon(w["text"]), canon(p)) if (w and p) else None
    sims[n] = s
    if w and p:
        use = "wikisource" if (s >= 0.90 and "…" not in w["text"]) else "pdf"
    elif w:
        use = "wikisource" if "…" not in w["text"] else "wikisource-unverified"
    elif p:
        use = "pdf"
    else:
        continue
    text = w["text"] if use.startswith("wikisource") else unwrap(p)
    chapter = w["chapter"] if w else last_chapter
    last_chapter = chapter
    final.append({"law": LAW3, "article_type": "law", "article": n,
                  "chapter": chapter, "text": text, "text_source": use})

vals = [s for s in sims.values() if s is not None]
bucket = lambda lo, hi: sum(lo <= s < hi for s in vals)
print("تشابه ≥0.97:", bucket(0.97, 2), "| 0.90-0.97:", bucket(0.90, 0.97),
      "| 0.75-0.90:", bucket(0.75, 0.90), "| <0.75:", bucket(0, 0.75))
print("المصدر:", Counter(r["text_source"] for r in final))
print("مواد من الـ PDF:", [r["article"] for r in final if r["text_source"] == "pdf"])
print("أقل 12 تشابه:", sorted((round(s, 2), n) for n, s in sims.items() if s is not None)[:12])
print("مواد 2 و20 و42 و193:", {n: round(sims[n], 2) for n in (2, 20, 42, 193) if sims.get(n) is not None})
print("مفقودة:", [n for n in range(1, 299) if n not in {r["article"] for r in final}])
print("إصدار فيها …:", sum("…" in r["text"] for r in issuance), "من", len(issuance))

for r in issuance:
    r["text_source"] = "wikisource"
labor = issuance + final          # ← دلوقتي labor هو النسخة المدمجة

تشابه ≥0.97: 128 | 0.90-0.97: 34 | 0.75-0.90: 16 | <0.75: 110
المصدر: Counter({'wikisource': 162, 'pdf': 136})
مواد من الـ PDF: [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 169, 170, 171, 172, 189, 190, 191, 192, 200, 236, 237, 238, 239, 240, 241, 250, 252, 253, 254, 275, 276, 277, 282, 283, 284, 288, 289, 290, 291, 293, 294, 296]
أقل 12 تشابه: [(0.02, 120), (0.02, 253), (0.02, 291), (0.03, 41), (0.03, 119), (0.03, 123), (0.04, 33), (0.04, 44), (0.04, 117), (0.04, 252), (0.06, 19), (0.06, 36)]
مواد 2 و20 و42 و193: {2: 0.6, 20: 0.12, 42: 0.2, 193: 0.96}
مفقودة: []
إصدار فيها …: 0 من 13


In [8]:
def art_label(r):
    return f"مادة الإصدار ({r['article']})" if r["article_type"] == "issuance" else f"المادة {r['article']}"

corpus = []
for src in (records, consumer, labor):
    for r in src:
        if not r["text"].strip():
            continue
        r = dict(r)
        r["id"] = f'{CODES[r["law"]]}-{r["article_type"]}-{r["article"]}'
        r["source"] = f'{r["law"]} - {art_label(r)}'
        corpus.append(r)

assert len({r["id"] for r in corpus}) == len(corpus), "فيه id مكرر"

(DATA / "processed").mkdir(exist_ok=True)
(DATA / "processed" / "corpus_articles.json").write_text(
    json.dumps(corpus, ensure_ascii=False, indent=1), encoding="utf-8")

print(len(corpus), "records |", sum(len(r["text"]) for r in corpus), "chars")
print(Counter(r["law"] for r in corpus))
lens = sorted((len(r["text"]), r["source"]) for r in corpus)
print("median:", lens[len(lens) // 2][0], "| max:", lens[-1][0])
print("أطول 5:", lens[-5:])

448 records | 212932 chars
Counter({'قانون العمل 14 لسنة 2025': 311, 'قانون حماية المستهلك 181 لسنة 2018': 81, 'قانون حماية البيانات الشخصية 151 لسنة 2020': 56})
median: 362 | max: 6694
أطول 5: [(2113, 'قانون حماية البيانات الشخصية 151 لسنة 2020 - المادة 5'), (2247, 'قانون حماية البيانات الشخصية 151 لسنة 2020 - المادة 19'), (2419, 'قانون حماية المستهلك 181 لسنة 2018 - المادة 1'), (4036, 'قانون حماية البيانات الشخصية 151 لسنة 2020 - المادة 1'), (6694, 'قانون العمل 14 لسنة 2025 - المادة 1')]


In [9]:
DIGRUN = re.compile(r"\d{2,}")
both = [n for n, s in sims.items() if s is not None and s >= 0.85]
same = rev = other = 0
ex = []
for n in both:
    a = Counter(DIGRUN.findall(wiki[n]["text"]))
    b = Counter(DIGRUN.findall(pdf_txt[n]))
    same += sum((a & b).values())
    for tok, c in (b - a).items():
        if tok != tok[::-1] and tok[::-1] in a:
            rev += c
            ex.append((n, tok, tok[::-1]))
        else:
            other += c

print("مواد قارنّاها:", len(both))
print("أرقام متطابقة:", same, "| مقلوبة في الـ PDF:", rev, "| اختلاف تاني:", other)
print("أمثلة:", ex[:12])

pdf_only = [r for r in final if r["text_source"] == "pdf"]
print("أرقام (خانتين+) في مواد الـ PDF:", sum(len(DIGRUN.findall(r["text"])) for r in pdf_only))
print("عينة سياقات:")
ctx = [m for r in pdf_only for m in re.findall(r".{0,28}\d{2,}.{0,14}", r["text"])]
for c in ctx[:14]:
    print(" •", c.replace("\n", " "))

مواد قارنّاها: 168
أرقام متطابقة: 104 | مقلوبة في الـ PDF: 1 | اختلاف تاني: 5
أمثلة: [(286, '25', '52')]
أرقام (خانتين+) في مواد الـ PDF: 204
عينة سياقات:
 • لمعاشات الصادر بالقانون رقم 148 لسنة 2019
 • 10- العامل فى ال
 • 11 - السخرة : كل
 • 12- الليل : الفت
 • 13- التوجيه المه
 • 14- التدريب: عمل
 • 15- التلمذة الصن
 • 16- مجالس المهار
 • 17- وكالات التشغ
 • 18- الوكلاء المف
 • 19- المفاوضة الج
 • 20- المنازعة الج
 • 21- الشركاء الاج
 • 22- المفوض العما


In [10]:
def smart_split_article(record, max_chars=1200):
    text = record["text"]
    if len(text) <= max_chars:
        return [record]

    # 1) قسّم على البنود المرقمة (1- ... 2- ...)
    items = re.split(r'\n(?=\d+\s*[-–\.]\s*)', text)

    # 2) لو مفيش بنود مرقمة (زي التعريفات): قسّم على "المصطلح : "
    if len(items) == 1:
        items = re.split(r'\n(?=[\u0621-\u064A\s]{2,30}\s*:\s*)', text)

    # 3) آخر حل: قسّم على الأسطر
    if len(items) == 1:
        items = re.split(r'\n+', text)

    sub_chunks = []
    curr_text = ""
    sub_idx = 1

    for item in items:
        item_str = item.strip()
        if not item_str:
            continue

        if len(curr_text) + len(item_str) <= max_chars:
            curr_text += ("\n" + item_str if curr_text else item_str)
        else:
            if curr_text:
                sub_rec = dict(record)
                sub_rec["id"] = f"{record['id']}-sub-{sub_idx}"
                sub_rec["text"] = curr_text
                sub_chunks.append(sub_rec)
                sub_idx += 1
            curr_text = item_str

    if curr_text:
        sub_rec = dict(record)
        sub_rec["id"] = f"{record['id']}-sub-{sub_idx}"
        sub_rec["text"] = curr_text
        sub_chunks.append(sub_rec)

    return sub_chunks


articles = json.loads((DATA / "processed" / "corpus_articles.json").read_text(encoding="utf-8"))

final_corpus = []
for record in articles:
    final_corpus.extend(smart_split_article(record, max_chars=1200))

(DATA / "processed" / "chunks.json").write_text(
    json.dumps(final_corpus, ensure_ascii=False, indent=1), encoding="utf-8")

print("articles:", len(articles), "-> chunks:", len(final_corpus))
longest = max(final_corpus, key=lambda x: len(x["text"]))
print("max text length:", len(longest["text"]), "|", longest["source"], "|", longest["id"])

articles: 448 -> chunks: 473
max text length: 1598 | قانون العمل 14 لسنة 2025 - المادة 253 | labor-law-253-sub-1


In [11]:
chunks = json.loads((DATA / "processed" / "chunks.json").read_text(encoding="utf-8"))

print(len(chunks), "chunks | unique ids:", len({c["id"] for c in chunks}))
print("empty:", sum(not c["text"].strip() for c in chunks),
      "| max len:", max(len(c["text"]) for c in chunks))
print("sub-chunks:", sum("-sub-" in c["id"] for c in chunks))

for law in CODES:
    sub = [c for c in chunks if c["law"] == law]
    print(law, "| chunks:", len(sub), "| فيها …:", sum("…" in c["text"] for c in sub))

labor_nums = {c["article"] for c in chunks if c["law"] == LAW3 and c["article_type"] == "law"}
print("labor missing:", sorted(set(range(1, 299)) - labor_nums))

print("labor text sources:", Counter(c.get("text_source") for c in chunks if c["law"] == LAW3))

473 chunks | unique ids: 473
empty: 0 | max len: 1598
sub-chunks: 44
قانون حماية البيانات الشخصية 151 لسنة 2020 | chunks: 64 | فيها …: 0
قانون حماية المستهلك 181 لسنة 2018 | chunks: 85 | فيها …: 0
قانون العمل 14 لسنة 2025 | chunks: 324 | فيها …: 0
labor missing: []
labor text sources: Counter({'wikisource': 178, 'pdf': 146})


In [12]:
import chromadb
from sentence_transformers import SentenceTransformer

EMB_NAME = "BAAI/bge-m3"
STORE = Path("backend/data/vector_store")
STORE.mkdir(parents=True, exist_ok=True)

chunks = json.loads((DATA / "processed" / "chunks.json").read_text(encoding="utf-8"))

def label(c):
    return f"مادة الإصدار ({c['article']})" if c["article_type"] == "issuance" else f"المادة {c['article']}"

def embed_text(c):      # العنوان بيتحط في النص المتشفّر بس، مش في النص اللي بيتعرض
    return f"{c['law']} | {c['chapter']} | {label(c)}\n{c['text']}"

model = SentenceTransformer(EMB_NAME, device="cpu")
vecs = model.encode([embed_text(c) for c in chunks], batch_size=16,
                    normalize_embeddings=True, show_progress_bar=True)

client = chromadb.PersistentClient(path=str(STORE))
try:
    client.delete_collection("laws")
except Exception:
    pass
col = client.create_collection("laws", metadata={"hnsw:space": "cosine"})
col.add(
    ids=[c["id"] for c in chunks],
    embeddings=vecs.tolist(),
    documents=[c["text"] for c in chunks],
    metadatas=[{"law": c["law"], "article": str(c["article"]), "article_type": c["article_type"],
                "chapter": c["chapter"], "source": c["source"],
                "text_source": c.get("text_source", "")} for c in chunks],
)

(STORE.parent / "rag_config.json").write_text(json.dumps({
    "embedding_model": EMB_NAME, "collection": "laws", "distance": "cosine",
    "chunking": "بنية المواد والبنود، بحد أقصى 1200 حرف، بدون overlap",
    "n_chunks": len(chunks),
}, ensure_ascii=False, indent=2), encoding="utf-8")

print("stored:", col.count())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

stored: 473


In [13]:
def retrieve(q, k=5):
    qv = model.encode([q], normalize_embeddings=True)
    r = col.query(query_embeddings=qv.tolist(), n_results=k)
    return [(m["source"].split(" - ")[-1], m["law"][:16], round(1 - d, 3))
            for m, d in zip(r["metadatas"][0], r["distances"][0])]

for q in ["خلال كام ساعة لازم أبلّغ عن خرق البيانات الشخصية؟",
          "هل أقدر أرجّع السلعة وأسترد فلوسي؟",
          "كم مدة الراحة الأسبوعية للعامل؟"]:
    print(q)
    for row in retrieve(q):
        print("  ", row)
        
def retrieve_full(q, k=5):
    qv = model.encode([q], normalize_embeddings=True)
    r = col.query(query_embeddings=qv.tolist(), n_results=k)
    return [{"text": doc, **meta, "score": round(1 - dist, 3)}
            for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]        

خلال كام ساعة لازم أبلّغ عن خرق البيانات الشخصية؟
   ('المادة 7', 'قانون حماية البي', 0.742)
   ('المادة 33', 'قانون حماية البي', 0.658)
   ('المادة 42', 'قانون حماية البي', 0.653)
   ('المادة 41', 'قانون حماية البي', 0.65)
   ('المادة 36', 'قانون حماية البي', 0.638)
هل أقدر أرجّع السلعة وأسترد فلوسي؟
   ('المادة 17', 'قانون حماية المس', 0.626)
   ('المادة 21', 'قانون حماية المس', 0.592)
   ('المادة 40', 'قانون حماية المس', 0.572)
   ('المادة 73', 'قانون حماية المس', 0.553)
   ('المادة 52', 'قانون حماية المس', 0.543)
كم مدة الراحة الأسبوعية للعامل؟
   ('المادة 120', 'قانون العمل 14 ل', 0.718)
   ('المادة 124', 'قانون العمل 14 ل', 0.647)
   ('المادة 128', 'قانون العمل 14 ل', 0.638)
   ('المادة 122', 'قانون العمل 14 ل', 0.635)
   ('المادة 54', 'قانون العمل 14 ل', 0.634)


In [ ]:
import ollama

def build_context(chunks):
    return "\n\n---\n\n".join(
        f"[مصدر {i}: {c['law']} - المادة {c['article']}]\n{c['text']}"
        for i, c in enumerate(chunks, 1))

SYSTEM_PROMPT = (
    "أنت مساعد قانوني متخصص في القوانين المصرية الثلاثة الموجودة في السياق: "
    "قانون حماية البيانات الشخصية، قانون حماية المستهلك، وقانون العمل.\n\n"
    "اتبع القواعد التالية بدقة:\n"
    "1. استخدم فقط النصوص الموجودة في Context، ولا تستخدم معلومات من خارجها.\n"
    "2. قبل الإجابة، حدد المادة الأكثر ارتباطًا بالسؤال.\n"
    "3. إذا كانت إحدى المواد تجيب عن السؤال مباشرة، استخدم هذه المادة كأساس للإجابة "
    "ولا تستبدلها بمادة تتناول حالة خاصة أو مختلفة.\n"
    "4. لا تدمج أحكام مادتين مختلفتين في حكم واحد.\n"
    "5. لا تضف أي شرط أو استثناء أو مدة أو حق إلا إذا كان مذكورًا صراحة في المادة "
    "التي تعتمد عليها.\n"
    "6. اذكر اسم القانون ورقم المادة بوضوح.\n"
    "7. إذا كانت هناك مواد أخرى مرتبطة بالسؤال، يمكنك ذكرها بشكل منفصل، "
    "لكن لا تخلط أحكامها مع المادة الأساسية.\n"
    "8. إذا لم يكن السياق كافيًا للإجابة، قل حرفيًا: "
    "'المعلومات المتاحة في النصوص المرفقة غير كافية للإجابة على هذا السؤال.'\n"
    "9. إذا كان السؤال خارج نطاق القوانين الثلاثة، قل حرفيًا: "
    "'هذا السؤال خارج نطاق القوانين المتاحة في النظام.'\n"
    "10. لا تخترع أي معلومات قانونية.\n"
    "11. أجب باللغة العربية وبشكل مختصر وواضح.")

REFUSAL = "هذا السؤال خارج نطاق القوانين المتاحة في النظام."
MIN_SCORE = 0.50

def ask_rag(query, k=4, model="qwen2.5:7b"):
    chunks = retrieve_full(query, k=k)
    if not chunks or chunks[0]["score"] < MIN_SCORE:
        return REFUSAL, chunks
    context = build_context(chunks)
    resp = ollama.chat(model=model, messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"السياق:\n{context}\n\nالسؤال: {query}"},
    ], options={"temperature": 0})
    return resp["message"]["content"], chunks

answer, sources = ask_rag("هل أقدر أرجّع السلعة وأسترد فلوسي؟")
print(answer)
for s in sources:
    print("-", s["source"])

نعم، يمكنك أرجاع السلعة واسترداد فلوسك طبقاً للمادة 17 من قانون حماية المستهلك 181 لسنة 2018، حيث يحق لك استبدال السلعة أو إعادتها مع استرداد قيمتها النقدية خلال أربعة عشر يوماً من تسلمها دون إبداء أسباب ودون تحمل أي نفقات.
- قانون حماية المستهلك 181 لسنة 2018 - المادة 17
- قانون حماية المستهلك 181 لسنة 2018 - المادة 21
- قانون حماية المستهلك 181 لسنة 2018 - المادة 40
- قانون حماية المستهلك 181 لسنة 2018 - المادة 73


In [21]:
for q in ["كم مدة الإجازة السنوية للعامل؟",
          "هل يجوز نقل البيانات الشخصية للخارج بدون ترخيص؟",
          "إزاي أعمل فراخ مشوية بالبيت؟"]:
    print("Q:", q)
    a, _ = ask_rag(q)
    print(a, "\n" + "-"*60)

Q: كم مدة الإجازة السنوية للعامل؟
وفقاً للمادة 125 من قانون العمل 14 لسنة 2025، يستحق العامل إجازة سنوية مدتها خمسة عشر يوماً على الأقل، مع حصول العامل على ستة أيام متصلة على الأقل. 
------------------------------------------------------------
Q: هل يجوز نقل البيانات الشخصية للخارج بدون ترخيص؟
لا يجوز نقل البيانات الشخصية للخارج بدون ترخيص حسب المادة 14 من قانون حماية البيانات الشخصية 151 لسنة 2020. 
------------------------------------------------------------
Q: إزاي أعمل فراخ مشوية بالبيت؟
هذا السؤال خارج نطاق القوانين المتاحة في النظام. 
------------------------------------------------------------


In [16]:
EVAL_SET = [
    # ---- قانون حماية البيانات الشخصية 151/2020 ----
    {"q": "خلال كام ساعة لازم المتحكم يبلّغ المركز عن خرق البيانات؟",
     "law": "pdp", "article": 7, "expected": "خلال 72 ساعة"},
    {"q": "هل ينفع أجمع بيانات شخصية حساسة عن الأطفال من غير موافقة ولي الأمر؟",
     "law": "pdp", "article": 12, "expected": "لا يجوز، يلزم موافقة ولي الأمر"},
    {"q": "إيه هي حقوق الشخص اللي بياناته بتتعالج؟",
     "law": "pdp", "article": 2, "expected": "العلم، العدول عن الموافقة، التصحيح، الاعتراض..."},
    {"q": "هل يجوز نقل البيانات الشخصية لدولة أجنبية من غير ترخيص؟",
     "law": "pdp", "article": 14, "expected": "لا، إلا بترخيص من المركز وتوافر مستوى حماية كافٍ"},
    {"q": "مين المسؤول عن حماية البيانات جوه الجهة، وإيه دوره؟",
     "law": "pdp", "article": 9, "expected": "مسؤول حماية البيانات، يراقب الالتزام ويتلقى الطلبات"},
    {"q": "إيه عقوبة جمع بيانات حساسة من غير ترخيص؟",
     "law": "pdp", "article": 41, "expected": "حبس 3 شهور وغرامة 500 ألف لـ 5 مليون جنيه"},

    # ---- قانون حماية المستهلك 181/2018 ----
    {"q": "هل أقدر أرجّع سلعة واسترد فلوسي من غير سبب؟",
     "law": "cpl", "article": 17, "expected": "نعم خلال 14 يوم من الاستلام دون إبداء أسباب"},
    {"q": "إيه ضمان السلع المعمرة زي الأجهزة الكهربائية؟",
     "law": "cpl", "article": 22, "expected": "سنتين على الأقل من الاستلام"},
    {"q": "هل يجوز للمحل يحبس سلعة عن البيع عشان يرفع السعر بعدين؟",
     "law": "cpl", "article": 8, "expected": "يُحظر حبس المنتجات الاستراتيجية عن التداول"},
    {"q": "إيه حقي لو اتصلت بشركة تسويق وأنا مش عايز أتلقى إعلانات منهم؟",
     "law": "cpl", "article": 17, "expected": "له حق رفض الاتصال والعدول عن الموافقة"},
    {"q": "في التعاقد عن بعد (أونلاين)، كام يوم أقدر أرجع في العقد؟",
     "law": "cpl", "article": 40, "expected": "14 يوم من استلام السلعة"},
    {"q": "مين المسؤول لو حصل ضرر بسبب عيب تصنيع في المنتج؟",
     "law": "cpl", "article": 27, "expected": "المنتج مسؤول، والموردون مسؤولية تضامنية"},

    # ---- قانون العمل 14/2025 ----
    {"q": "كم مدة الراحة الأسبوعية للعامل؟",
     "law": "labor", "article": 120, "expected": "24 ساعة كاملة بعد 6 أيام عمل متصلة"},
    {"q": "إيه الحد الأقصى لساعات العمل في اليوم؟",
     "law": "labor", "article": 119, "expected": "لا تجاوز الفترة بين بداية العمل ونهايته 10 ساعات"},
    {"q": "متى يعتبر عقد العمل محدد المدة عقد غير محدد المدة؟",
     "law": "labor", "article": 88, "expected": "لو غير مكتوب، أو مالوش مدة، أو استمر التنفيذ بعد انتهاء مدته"},
    {"q": "هل المنشآت لازم توفر حضانة للعاملين؟",
     "law": "labor", "article": 60,
     "expected": "المنشأة التي تستخدم مائة عاملة فأكثر في مكان واحد تلتزم بإنشاء دار حضانة أو العهد إلى دار حضانة برعاية أطفال العاملات"},
    {"q": "إيه هي حالات فصل العامل بدون إخطار أو مكافأة؟",
     "law": "labor", "article": None, "expected": "الأخطاء الجسيمة (تُراجع بحسب المادة الدقيقة في القانون)"},
    {"q": "إزاي بتتم المفاوضة الجماعية بين العمال وأصحاب الأعمال؟",
     "law": "labor", "article": 194, "expected": "بحرية وطواعية لتحسين شروط العمل وتسوية المنازعات"},

    # ---- برّه النطاق (اختبار الرفض) ----
    {"q": "إزاي أعمل كشري بالبيت؟",
     "law": None, "article": None, "expected": "رفض، غير موجود في السياق"},
    {"q": "إيه عقوبة القتل العمد في القانون المصري؟",
     "law": None, "article": None, "expected": "رفض، برّه نطاق القوانين الثلاثة"},
]
print(len(EVAL_SET), "سؤال")

20 سؤال


In [17]:
for item in EVAL_SET:
    chunks = retrieve_full(item["q"], k=1)
    print(round(chunks[0]["score"], 3), "|", item["q"][:40])

0.706 | خلال كام ساعة لازم المتحكم يبلّغ المركز 
0.687 | هل ينفع أجمع بيانات شخصية حساسة عن الأطف
0.703 | إيه هي حقوق الشخص اللي بياناته بتتعالج؟
0.711 | هل يجوز نقل البيانات الشخصية لدولة أجنبي
0.611 | مين المسؤول عن حماية البيانات جوه الجهة،
0.681 | إيه عقوبة جمع بيانات حساسة من غير ترخيص؟
0.648 | هل أقدر أرجّع سلعة واسترد فلوسي من غير س
0.672 | إيه ضمان السلع المعمرة زي الأجهزة الكهرب
0.526 | هل يجوز للمحل يحبس سلعة عن البيع عشان ير
0.581 | إيه حقي لو اتصلت بشركة تسويق وأنا مش عاي
0.72 | في التعاقد عن بعد (أونلاين)، كام يوم أقد
0.686 | مين المسؤول لو حصل ضرر بسبب عيب تصنيع في
0.718 | كم مدة الراحة الأسبوعية للعامل؟
0.676 | إيه الحد الأقصى لساعات العمل في اليوم؟
0.742 | متى يعتبر عقد العمل محدد المدة عقد غير م
0.718 | هل المنشآت لازم توفر حضانة للعاملين؟
0.62 | إيه هي حالات فصل العامل بدون إخطار أو مك
0.639 | إزاي بتتم المفاوضة الجماعية بين العمال و
0.479 | إزاي أعمل كشري بالبيت؟
0.554 | إيه عقوبة القتل العمد في القانون المصري؟


In [18]:
import time

results = []
for i, item in enumerate(EVAL_SET, 1):
    chunks = retrieve_full(item["q"], k=4)
    hit = any(str(c.get("article")) == str(item["article"]) for c in chunks) if item["article"] else None
    t0 = time.time()
    answer, _ = ask_rag(item["q"], k=4)
    dt = round(time.time() - t0, 1)
    results.append({"i": i, "q": item["q"], "expected_law": item["law"],
                    "expected_article": item["article"], "hit_at_4": hit,
                    "answer": answer, "seconds": dt})
    print(f"[{i}/{len(EVAL_SET)}] hit={hit} ({dt}s)")

import pandas as pd
df = pd.DataFrame(results)
print("Hit@4 rate (للأسئلة اللي ليها مادة محددة):",
      round(df[df.expected_article.notna()]["hit_at_4"].mean(), 2))
df.to_csv(DATA / "processed" / "evaluation_results.csv", index=False, encoding="utf-8-sig")
df[["i", "q", "expected_article", "hit_at_4", "seconds"]]

[1/20] hit=True (5.6s)
[2/20] hit=True (5.7s)
[3/20] hit=True (22.4s)
[4/20] hit=True (7.0s)
[5/20] hit=True (27.9s)
[6/20] hit=True (6.7s)
[7/20] hit=True (7.4s)
[8/20] hit=True (7.4s)
[9/20] hit=True (8.2s)
[10/20] hit=True (7.6s)
[11/20] hit=True (6.3s)
[12/20] hit=True (10.8s)
[13/20] hit=True (7.0s)
[14/20] hit=True (6.1s)
[15/20] hit=True (7.1s)
[16/20] hit=True (5.7s)
[17/20] hit=None (33.9s)
[18/20] hit=True (17.0s)
[19/20] hit=None (0.2s)
[20/20] hit=None (3.2s)
Hit@4 rate (للأسئلة اللي ليها مادة محددة): 1.0


,i,q,expected_article,hit_at_4,seconds
0,1,خلال كام ساعة لازم المتحكم يبلّغ المركز عن خرق...,7.0,True,5.6
1,2,هل ينفع أجمع بيانات شخصية حساسة عن الأطفال من ...,12.0,True,5.7
2,3,إيه هي حقوق الشخص اللي بياناته بتتعالج؟,2.0,True,22.4
3,4,هل يجوز نقل البيانات الشخصية لدولة أجنبية من غ...,14.0,True,7.0
4,5,مين المسؤول عن حماية البيانات جوه الجهة، وإيه ...,9.0,True,27.9
5,6,إيه عقوبة جمع بيانات حساسة من غير ترخيص؟,41.0,True,6.7
6,7,هل أقدر أرجّع سلعة واسترد فلوسي من غير سبب؟,17.0,True,7.4
7,8,إيه ضمان السلع المعمرة زي الأجهزة الكهربائية؟,22.0,True,7.4
8,9,هل يجوز للمحل يحبس سلعة عن البيع عشان يرفع الس...,8.0,True,8.2
9,10,إيه حقي لو اتصلت بشركة تسويق وأنا مش عايز أتلق...,17.0,True,7.6


In [19]:
answer, sources = ask_rag("هل أقدر أرجّع السلعة وأسترد فلوسي؟")

print(answer)

print("\nالمصادر:")
for s in sources:
    print("-", s["source"])

نعم، يمكنك رجوع السلعة واسترداد قيمة الفلوس خلال أربعة عشر يوما من تسلمها، وفقاً للمادة 17 من قانون حماية المستهلك 181 لسنة 2018.

المصادر:
- قانون حماية المستهلك 181 لسنة 2018 - المادة 17
- قانون حماية المستهلك 181 لسنة 2018 - المادة 21
- قانون حماية المستهلك 181 لسنة 2018 - المادة 40
- قانون حماية المستهلك 181 لسنة 2018 - المادة 73


In [20]:
print(f"""
تحليل الفشل - Evaluation Summary
=================================
إجمالي الأسئلة: {len(df)}
Hit@4 (للأسئلة ذات المادة المحددة): {round(df[df.expected_article.notna()]['hit_at_4'].mean(), 2)}
متوسط زمن الاستجابة: {round(df['seconds'].mean(), 1)} ثانية

الحالات اللي تحتاج مراجعة يدوية: أسئلة برّه النطاق أو بدون مادة محددة (i=17,19,20)
""")
print(df[df["i"].isin([17, 19, 20])][["i", "q", "answer"]].to_string())


تحليل الفشل - Evaluation Summary
إجمالي الأسئلة: 20
Hit@4 (للأسئلة ذات المادة المحددة): 1.0
متوسط زمن الاستجابة: 10.2 ثانية

الحالات اللي تحتاج مراجعة يدوية: أسئلة برّه النطاق أو بدون مادة محددة (i=17,19,20)

     i                                              q                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 